# Integrate datasets

_Part of the **Decay Before Archival** project — see `README.md` for full project context, setup instructions, and links to the per-source notebooks (`deps_dev.ipynb`, `scorecard.ipynb`, `gh_archive.ipynb`, `pypi.ipynb`, `other_sources.ipynb`)._

**Purpose:** this is where we combine the per-source explorations into one dataset once each notebook has settled on the fields it wants to keep. It stays a placeholder until that's ready — there's nothing to merge yet.

**Before filling this in:**
- Each source notebook should have decided what it exports per project/package (and under what column name — e.g. `repo_owner`/`repo_name`, or a full repo URL) so we have a consistent join key across sources.
- Merging happens here, not in the individual notebooks — keep source notebooks focused on exploring their own dataset.

In [ ]:
from google.cloud import bigquery
from decay_before_archival import get_client

client = get_client()

## Join keys per source

| Source | Join key column(s) | Notes |
|---|---|---|
| GH Archive (base table) | `repo_full_name` (`repo_id` is also stable) | 62,627 repos (62,581 unique names); every other source is left-joined onto this |
| deps.dev | `repo_full_name_lower` | matched on `LOWER(repo_full_name)`; only repos known to deps.dev appear |
| OpenSSF Scorecard | `repo_name` | already lowercase, one row per GH Archive name; latest score before the 2025-01-01 cutoff |
| PyPI / npm downloads | — | not collected; see `package_registries.ipynb` for the decision |


In [ ]:
import pandas as pd

gh = pd.read_csv("../data/gh_archive_results_clean.csv")
gh["repo_key"] = gh["repo_full_name"].str.lower()

deps = pd.read_csv("../data/deps_dev_features_2024.csv").rename(columns={"repo_full_name_lower": "repo_key"})
print(f"GH Archive: {len(gh):,} rows ({gh['repo_key'].nunique():,} unique keys) | deps.dev: {len(deps):,} rows ({deps['repo_key'].nunique():,} unique keys)")

**Notes:**

GH Archive contains 62,627 rows with 62,581 unqiue keys where deps.dev contains 1,405 rows with the same number of unique keys

## Merge (TODO)

Combine the sources on the join key(s) above (e.g. `pandas.merge`, or a BigQuery `JOIN` if it's cleaner to do it in SQL).

In [ ]:
# Left join: keep every GH Archive repo, add deps.dev columns where we have them.
merged = gh.merge(deps.assign(in_depsdev=1), on="repo_key", how="left", validate="m:1")

# Indicators: a repo missing from deps.dev is structurally missing, not randomly missing.
merged["in_depsdev"] = merged["in_depsdev"].fillna(0).astype(int)
merged["has_package"] = (merged["n_packages"].fillna(0) > 0).astype(int)
merged["in_projects"] = merged["in_projects"].fillna(False).astype(int)  # not in deps.dev => not in Projects
merged["days_since_last_version_at_cutoff_missing"] = merged["days_since_last_version_at_cutoff"].isna().astype(int)

# Package/version counts are true zeros for repos with no package; stars/forks/issues stay NaN (unknown, not zero).
count_cols = ["n_packages", "n_ecosystems", "n_versions_total", "versions_2024", "versions_2023", "advisory_links"]
merged[count_cols] = merged[count_cols].fillna(0)
merged.shape


### Join Scorecard

`scorecard_gh_sample.csv` already has one row per GH Archive name (latest score before the cutoff, `NaN` if Scorecard never scored the repo). The date is turned into an age so it can be used as a feature.

In [ ]:
sc = pd.read_csv("../data/scorecard_gh_sample.csv", parse_dates=["scorecard_date"]).rename(columns={"repo_name": "repo_key"})
merged = merged.merge(sc, on="repo_key", how="left", validate="m:1")

merged["in_scorecard"] = merged.pop("has_scorecard_match").fillna(False).astype(int)
# Score stays NaN where unscored (structural, not imputed); date becomes age at the cutoff.
merged["days_since_scorecard_at_cutoff"] = (pd.Timestamp("2025-01-01") - merged.pop("scorecard_date")).dt.days
merged.shape


**Notes:**

- Table A keeps all 62,627 GH Archive rows (49 columns); both joins are many-to-one on the lowercase name, so no rows are added or dropped.
- Indicator columns mark *which source* a value came from: `in_depsdev`, `has_package`, `in_projects`, `in_scorecard`. Missing deps.dev and Scorecard values are structural (the repo isn't in that source), so they stay `NaN` and are not imputed; package/version counts are filled with 0 because "no package" really means zero.
- Scorecard dates are all before the cutoff (279 of 287 on 2024-12-30, 6 from 2021), so there is no leakage; `days_since_scorecard_at_cutoff` keeps the stale ones visible.

## Match rate (TODO)

Report how many rows matched across sources vs. how many were dropped, so we know how much coverage loss the join introduced.

In [ ]:
n = len(gh)
pkgs = pd.read_csv("../data/deps_dev_package_names_2024.csv")
npm_pypi = set(pkgs.loc[pkgs["System"].isin(["NPM", "PYPI"]), "repo_full_name_lower"])

stages = {
    "GH Archive repos (base)": n,
    "matched in deps.dev": int(merged["in_depsdev"].sum()),
    "matched AND has a package": int(merged["has_package"].sum()),
    "has an npm or PyPI package": int(merged["repo_key"].isin(npm_pypi).sum()),
    "matched in Scorecard (has a score)": int(merged["in_scorecard"].sum()),
    "matched in deps.dev AND Scorecard": int((merged["in_depsdev"] & merged["in_scorecard"]).sum()),
}
match = pd.DataFrame({"rows": stages}).rename_axis("stage")
match["lost"] = n - match["rows"]
match["pct_of_base"] = (100 * match["rows"] / n).round(2)
print(match.to_string())

# Rows on the other side that found no GH Archive repo (should all be 0).
print("\nunmatched source rows:", {
    "deps.dev": len(set(deps["repo_key"]) - set(gh["repo_key"])),
    "Scorecard": len(set(sc["repo_key"]) - set(gh["repo_key"])),
})

# Missing values per column, overall and among matched rows only (separates join-caused gaps from real nulls).
m = merged.loc[merged["in_depsdev"] == 1]
missing = pd.DataFrame({
    "missing_all": merged.isna().sum(),
    "pct_all": (100 * merged.isna().mean()).round(2),
    "missing_in_depsdev": m.isna().sum(),
    "pct_in_depsdev": (100 * m.isna().mean()).round(2),
})
missing[missing["missing_all"] > 0].sort_values("missing_all", ascending=False)


In [ ]:
# Does matching correlate with the label? (inactive_next_365d rate by source coverage)
display(merged.groupby(["in_depsdev", "has_package"])["inactive_next_365d"].agg(rows="size", inactive_rate="mean").round(3))
merged.groupby("in_scorecard")["inactive_next_365d"].agg(rows="size", inactive_rate="mean").round(3)


In [ ]:
# Table A: GH Archive + deps.dev + Scorecard, one row per GH Archive repo.
WRITE_MERGED = True

if WRITE_MERGED:
    merged.to_csv("../data/merged_dataset.csv", index=False)
    print(f"wrote {len(merged):,} rows x {merged.shape[1]} cols")


**Notes:**

- **Match rate:**

  | Stage | Rows | % of base |
  |---|---|---|
  | GH Archive repos (base) | 62,627 | 100.00 |
  | matched in deps.dev | 1,405 | 2.24 |
  | matched AND has a package | 951 | 1.52 |
  | has an npm or PyPI package | 480 | 0.77 |
  | matched in Scorecard (has a score) | 287 | 0.46 |
  | matched in deps.dev AND Scorecard | 287 | 0.46 |

  Every deps.dev and Scorecard row matches a GH Archive repo, so nothing was dropped on the source side. Every Scorecard repo is also in deps.dev: both sources cover the same small, popular corner of the sample.
- **Inactive rate (`inactive_next_365d`) by deps.dev status:**

  | Group | Repos | Inactive rate |
  |---|---|---|
  | Not in deps.dev | 61,222 | 85.6% |
  | In deps.dev, no package | 454 | 63.9% |
  | Has a package | 951 | 44.9% |
  | Has a Scorecard score | 287 | 25.4% |

  This is a correlation, not a cause: published packages tend to be more mature and popular (the popularity bias the proposal flags). The 85.6% base rate also says most of this sample is small and quiet — worth checking against how the GH Archive sample was drawn.
- **Missingness:** `stars_end` / `forks_end` / `open_issues_end` are missing for 165 matched rows (package-only repos); the `_start` columns for 854. `scorecard_score` is missing for 99.54% overall and 79.6% of deps.dev repos. These gaps come from the join, so use the `in_*` / `has_package` indicators rather than imputing. `days_since_last_push_at_cutoff` is the only GH Archive column with real gaps (1.9% overall).
- **Duplicate names:** 79 rows share 33 names across different `repo_id`s; none match deps.dev or Scorecard, so they are kept (see the reviewer decision below).
- **Output:** `data/merged_dataset.csv` (Table A): 62,627 rows x 49 columns.

## Open decision: how to join GitHub API data (for team review)

**Problem.** The GitHub API dataset (`data/github_metrics*.csv`) covers 10,330 repos drawn from a pool of repos *created in 2024-2025* and collected live in 2026. Our other sources use the GH Archive sample with a 2024 cutoff and the `inactive_next_365d` label. The two samples barely overlap (name match, lowercased):

| Source | Repos | Overlap with the GitHub API 10,330 |
|---|---|---|
| GH Archive (base table) | 62,581 | **15** |
| Scorecard | 62,581 (287 with a score) | 15 |
| deps.dev | 1,405 | 4 |

A left join onto GH Archive would therefore fill the GitHub API columns for about 0.02% of rows. This is a sampling-frame mismatch, not a name-matching problem.

**Options**

1. **Left join onto GH Archive anyway.** Matches the notebook as written; columns would be almost entirely missing. Not recommended.
2. **Use 10,330 repos as the base table** and left join the other sources onto it. Coverage from GH Archive, Scorecard and deps.dev would be tiny, and we would lose the `inactive_next_365d` label.
3. **Two merged tables.** (a) GH Archive + deps.dev + Scorecard, the main modelling set with the label. (b) the GitHub API pool analysed separately (descriptive and EDA, or as a validation set). Report the 15-repo overlap as the match-rate finding. *Recommended starting point.*
4. **Collect metrics for the GH Archive repos** (or a stratified subset), so the sources really overlap. The only option that gives a usable joint dataset, but it costs GitHub API quota and time.

Options 3 and 4 combine: build table (a) now and pursue 4 in parallel if the team wants his metrics in the model.

**Same root cause, deps.dev.** deps.dev only lists repos linked to a published package, and the GH Archive sample is mostly tiny repos (`stars_received_2024`: median 0, 99th percentile 12), so only 1,405 of 62,627 rows match. A same-name/different-owner search found only forks and personal copies, not renames, so the exact lowercase name join stays. A larger deps.dev set would need a different base sample (start from packaged repos, or filter GH Archive by minimum activity).

**Other items to settle before the merge**

- ~~Stale GH Archive path~~ — fixed: now reads `data/gh_archive_results_clean.csv` (same 62,627 rows; match-rate numbers unchanged).
- ~~Scorecard not joined~~ — done: joined on `repo_name` (lowercase), one row per GH Archive repo; only 0.46% have a score.
- ~~46 duplicate GH Archive repo names~~ — resolved: keep them (see reviewer decision below).
- keys are `repo_full_name` (mixed case, no duplicates), so lowercase them like the others.
- His `decay_stage` / `archetype` columns are derived by his EDA notebook; use the raw metrics unless the team wants those definitions.

**Cleaning order.** Do per-source cleaning (types, key normalization, exact duplicates) in each source notebook before merging. Leave decisions that depend on the merged set (imputation, dropping rows, duplicate-name policy) until after the join, because missingness here is structural and the right handling depends on the model.

_Reviewers: add your choice and reasoning below._

### Reviewer decision (teammate)

**Choice: Option 3** (two merged tables), ideally combined with Option 4 (collect GitHub API metrics for the GH Archive repos) if time allows. Given limited time, start with Option 3 alone and come back to Option 4 later.

**Duplicate repo names:** the teammate thinks these are separate repos, since each has its own unique `repo_id`. Checked against `data/gh_archive_results_clean.csv`:

- It is 79 rows sharing 33 names (46 extra rows), and `repo_id` is unique, so they are distinct repos (deleted and recreated under the same name).
- Mostly bot or test accounts (`atlasbuildbot`, `ghc-automation-staging-...`, `pixeegithub-testuser4`, `whiteprints-tests`), 0 stars received in 2024.
- None of the 33 names appear in deps.dev or have a Scorecard score, so the name join copies no values onto the wrong repo.

**Decision: keep them, no dedupe.** The join is unaffected. Whether to filter bot/test repos is a separate modelling question for the whole sample, not specific to these 46.

## Table B: GitHub API repos (Option 3b)

The GitHub API snapshot (10,330 repos created 2024-2025, collected in 2026) is a different sample from GH Archive, so it is kept as its own table instead of being joined onto Table A. `pool.csv` adds the repo attributes from the search step (`archived`, `language`, `license_spdx`, stars). It has no `inactive_next_365d` label; `archived` is the outcome here.

In [ ]:
api = pd.read_csv("../data/github_metrics_snapshot.csv", parse_dates=["last_commit_at", "last_release_at"])
pool = pd.read_csv("../data/pool.csv", usecols=["repo_full_name", "archived", "language", "license_spdx", "stargazers_count", "created_at"])
table_b = api.merge(pool, on="repo_full_name", how="left", validate="1:1", indicator=True)
print(table_b["_merge"].value_counts().to_string())
table_b = table_b.drop(columns="_merge")
table_b["repo_key"] = table_b["repo_full_name"].str.lower()

# The collector's timezone quirk gives a few negative ages; a commit can't be in the future, so clip to 0.
for c in ["days_since_last_commit", "days_since_last_release"]:
    print(f"{c}: {(table_b[c] < 0).sum()} negative values clipped to 0")
    table_b[c] = table_b[c].clip(lower=0)

overlap = set(table_b["repo_key"]) & set(merged["repo_key"])
print(f"\nTable B: {len(table_b):,} repos | archived: {int(table_b['archived'].sum())} ({table_b['archived'].mean():.1%}) | overlap with Table A: {len(overlap)}")
table_b.isna().sum().loc[lambda s: s > 0].to_frame("missing").assign(pct=lambda d: (100 * d["missing"] / len(table_b)).round(2))


In [ ]:
table_b.to_csv("../data/github_api_table.csv", index=False)
print(f"wrote {len(table_b):,} rows x {table_b.shape[1]} cols")


**Notes:**

- **Match rate:** all 10,330 snapshot repos match `pool.csv` (1:1), so nothing is lost inside Table B. Only **15** of them are in Table A, which is why the two tables are analysed separately.
- **Outcome:** 317 repos (3.1%) are archived.
- **Cleaning:** 70 negative `days_since_last_commit` and 10 negative `days_since_last_release` values (collector timezone quirk) clipped to 0.
- **Missingness (all structural):**
  - `last_release_at` / `days_since_last_release` 40.9%: the repo never published a release. Treat as its own category ("no release") rather than imputing; the archived rate is similar with or without a release (2.9% vs 3.2%).
  - `bus_factor` 0.5% (55 repos): all have `n_contributors_top100 = 0`, so there is no one to compute it from.
  - `language` 4.8%: GitHub detected no language (e.g. docs-only repos); keep as an "unknown" category.
- **Output:** `data/github_api_table.csv` (Table B): 10,330 rows x 17 columns.